# N7 Tube Architecture Sweep

Runs the tube architecture comparison using the core settings from `2d_tube_diag.ipynb`. Outputs include per-run artifacts, paper-ready architecture comparison plots, regenerated energy landscapes, and post-training Hessian diagnostics.

In [1]:
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import TubeN7Properties
from run_architectures import SweepConfig

ROOT = Path.cwd()
train_file = "../experiment_data/n7_tube_train_dataset.npz"
valid_file = "../experiment_data/n7_tube_test_dataset.npz"
properties = TubeN7Properties(mass=0.005)

# Copied from 2d_tube_diag.ipynb
K_init_diag = (0.002, 0.005)
K_init_chol = (0.002, 0.0, 0.005)

base_cfg = SweepConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10,),
    corr_factor=1.0,
    input_mode="raw",
    only_stretching_NN=False,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    mode="isotropic",
    n_epochs=1000,
    lr=5e-3,
    weight_decay=0.0,
    seed=42,
    valid_every=10,
    max_dlambda=5e-3,
    iters=10,
    ls_steps=10,
    abs_tol=5e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=False,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-6,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key=None,
    force_loss_strength=0.0,
    force_components=(0, 1, 2),
    force_sign=1.0,
    return_loss_components=True,
    early_stopping=False,
    # early_stopping_patience=200,
    # early_stopping_min_delta=1e-5,
    # early_stopping_warmup_epochs=200,
    restore_best_model=True,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=False,
    plot_force_predictions=False,
    save_hessian_diagnostics=False,
    save_energy_landscapes=False,
    verbose=True,
    continue_on_failure=True,
)

print(properties)

OUTPUT_DIR = ROOT / "arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping"
PAPER_PLOT_DIR = OUTPUT_DIR / "paper_ready_architecture_comparison"
base_cfg = base_cfg.__class__(**{**base_cfg.__dict__, "output_dir": str(OUTPUT_DIR)})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving architecture sweep results under: {OUTPUT_DIR.resolve()}")


TubeN7Properties(length=None, r0=0.005, axs=None, jxs=None, ixs1=None, ixs2=None, density=600.0, E=1000000.0, N=7, start=Array([0., 0., 0.], dtype=float64), end=Array([0.46, 0.  , 0.03], dtype=float64), mass=0.005)
Saving architecture sweep results under: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping


In [2]:
from run_architectures import subset_tape_tube_candidates, subset_brazier_stiffness_only

selected_architectures = subset_tape_tube_candidates()

# For a quicker debug pass, use only the models closest to 2d_tube_diag.ipynb:
# selected_architectures = [
#     "brazier_diag_stiffness_baseline",
#     "brazier_diag_stiffness_mlp",
#     "brazier_diag_stiffness_icnn",
# ]

print(f"Running {len(selected_architectures)} architectures:")
for name in selected_architectures:
    print(f"  - {name}")
print("\nPer-architecture plots:", base_cfg.save_plots)
print("Energy landscape snapshots during training:", base_cfg.save_energy_landscapes)


Running 14 architectures:
  - diag_energy_baseline
  - diag_energy_mlp
  - diag_energy_icnn
  - mlp_energy
  - icnn_energy
  - chol_energy_baseline
  - chol_energy_mlp
  - chol_energy_icnn
  - brazier_diag_stiffness_baseline
  - brazier_diag_stiffness_mlp
  - brazier_diag_stiffness_icnn
  - brazier_chol_stiffness_baseline
  - brazier_chol_stiffness_mlp
  - brazier_chol_stiffness_icnn

Per-architecture plots: True
Energy landscape snapshots during training: False


In [3]:
from run_architectures import run_architecture_sweep

results = run_architecture_sweep(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=base_cfg,
    selected_architectures=selected_architectures,
)

successes = [name for name, result in results.items() if result["success"]]
failures = {name: result["failure_reason"] for name, result in results.items() if not result["success"]}

print(f"Succeeded: {len(successes)}/{len(results)}")
if failures:
    print("Failures:")
    for name, reason in failures.items():
        print(f"  - {name}: {reason}")
else:
    print("No architecture failures.")


Running architecture: diag_energy_baseline
  model_cls               : DiagonalPlusEnergyNN
  which_case              : baseline
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 1.0
  zero_reference          : True
  seed                    : 42
  n_epochs                : 1000
  lr                      : 0.005
  weight_decay            : 0.0
  max_dlambda             : 0.005
  iters                   : 10
  ls_steps                : 10
  abs_tol                 : 0.0005
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : False
  early_stopping_patience : 25
  early_stopping_warmup   : 0
  restore_best_model      : True
  hessian_reg_strength    : 1e-06
  he

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Running architecture: diag_energy_mlp
  model_cls               : DiagonalPlusEnergyNN
  which_case              : MLP
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 1.0
  zero_reference          : True
  seed                    : 42
  n_epochs                : 1000
  lr                      : 0.005
  weight_decay            : 0.0
  max_dlambda             : 0.005
  iters                   : 10
  ls_steps                : 10
  abs_tol                 : 0.0005
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : False
  early_stopping_patience : 25
  early_stopping_warmup   : 0
  restore_best_model      : True
  hessian_reg_strength    : 1e-06
  hessian_reg_

In [4]:
from run_architectures import generate_energy_landscapes_for_sweep

energy_landscape_paths = generate_energy_landscapes_for_sweep(
    str(OUTPUT_DIR),
    architectures=successes if "successes" in globals() else selected_architectures,
    include_initial=False,
    include_final=True,
    use_valid=True,
    traj_idx=0,
    dpi=180,
    n_grid=None,
    continue_on_failure=True,
)
print("Energy landscape plots regenerated for", len(energy_landscape_paths), "runs.")


[ok] brazier_chol_stiffness_baseline: {'final': '/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping/brazier_chol_stiffness_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_1__zr1__seed_42__mdl_0.005__it_10__hreg_1e-06__hprobe_1__hseed_0__mode_isotropic/energy_landscapes/energy_landscape_final.png'}
[ok] brazier_chol_stiffness_icnn: {'final': '/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping/brazier_chol_stiffness_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_1__zr1__seed_42__mdl_0.005__it_10__hreg_1e-06__hprobe_1__hseed_0__mode_isotropic/energy_landscapes/energy_landscape_final.png'}
[ok] brazier_chol_stiffness_mlp: {'final': '/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on

In [5]:
from architecture_plots import plot_architecture_comparison_paper, plot_summary_final_losses

PAPER_PLOT_DIR.mkdir(parents=True, exist_ok=True)
paper_architectures = successes if "successes" in globals() else selected_architectures
paper_paths = plot_architecture_comparison_paper(
    architectures=paper_architectures,
    results_dir=str(OUTPUT_DIR),
    output_dir=str(PAPER_PLOT_DIR),
    traj_idx=0,
)
plot_summary_final_losses(
    {name: results[name] for name in paper_architectures},
    save_path=str(PAPER_PLOT_DIR / "final_loss_summary.png"),
    show=False,
)
print("Paper-ready plots written to:", PAPER_PLOT_DIR.resolve())
for key, path in paper_paths.items():
    if key != "colors":
        print(f"  {key}: {path}")


Paper-ready plots written to: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping/paper_ready_architecture_comparison
  training_loss: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping/paper_ready_architecture_comparison/training_loss_comparison.pdf
  validation_loss: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping/paper_ready_architecture_comparison/validation_loss_comparison.pdf
  training_trajectory: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping/paper_ready_architecture_comparison/training_trajectory_comparison.pdf
  validation_trajectory: /Users/radha/Gi

In [6]:
import subprocess
import sys

HESSIAN_USE_PREDICTED = True
HESSIAN_STRIDE = 10
HESSIAN_MAX_TRAJECTORIES = 1
HESSIAN_SPLITS = ("train", "valid")

cmd = [
    sys.executable,
    "compute_architecture_hessian_diagnostics.py",
    str(OUTPUT_DIR),
    "--stride", str(HESSIAN_STRIDE),
    "--splits", *HESSIAN_SPLITS,
]
if HESSIAN_USE_PREDICTED:
    cmd.append("--use-predicted")
if HESSIAN_MAX_TRAJECTORIES is None:
    cmd.append("--all-trajectories")
else:
    cmd.extend(["--max-trajectories", str(HESSIAN_MAX_TRAJECTORIES)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)
print("Hessian diagnostics table:", OUTPUT_DIR / "hessian_diagnostics_table.csv")


Running: /Users/radha/GitRepos/dismech-jax/.venv/bin/python compute_architecture_hessian_diagnostics.py /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping --stride 10 --splits train valid --use-predicted --max-trajectories 1
[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping/brazier_chol_stiffness_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_1__zr1__seed_42__mdl_0.005__it_10__hreg_1e-06__hprobe_1__hseed_0__mode_isotropic | train: M=2.630e+02, kappa=1.622e+04, states=2 | valid: M=1.730e+02, kappa=7.227e+03, states=1
[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n7_tube_all_architectures_train_fail_on_nonconvergence_False_no_early_stopping/brazier_chol_stiffness_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh

In [7]:
print("Done. Key outputs:")
print("  - <architecture>/results.npz")
print("  - <architecture>/model.eqx")
print("  - <architecture>/hessian_diagnostics_summary.json")
print("  - hessian_diagnostics_table.csv")
print("  - paper_ready_architecture_comparison/*.pdf")


Done. Key outputs:
  - <architecture>/results.npz
  - <architecture>/model.eqx
  - <architecture>/hessian_diagnostics_summary.json
  - hessian_diagnostics_table.csv
  - paper_ready_architecture_comparison/*.pdf
